Auswertung EGIDS von Projektierten Gebäuden

Enviorment

In [ ]:
# conda activate ProjtoEGID

Download Projektierte Gebäude aus AV Daten

In [1]:
import requests
import geopandas as gpd
from io import BytesIO
from shapely.geometry import shape
import pandas as pd
from tqdm import tqdm

# WFS connection
BASE_URL = "https://geodienste.ch/db/av_0/deu"

params = {
    "SERVICE": "WFS",
    "VERSION": "2.0.0",
    "REQUEST": "GetFeature",
    "TYPENAMES": "ms:LCSFPROJ",
    "OUTPUTFORMAT": "application/json; subtype=geojson",
    "SRSNAME": "EPSG:2056",
    "COUNT": 10000,
    "STARTINDEX": 0,
}

# Paged fetching
all_gdfs = []
with tqdm(desc="Fetching WFS pages") as pbar:
    while True:
        response = requests.get(BASE_URL, params=params)
        gdf_page = gpd.read_file(BytesIO(response.content))
        if gdf_page.empty:
            break
        all_gdfs.append(gdf_page)
        params["STARTINDEX"] += params["COUNT"]
        pbar.update(len(gdf_page))

# Combine and save
gdf = pd.concat(all_gdfs, ignore_index=True)
gdf.to_file("av_projected_buildings.gpkg", driver="GPKG", layer="LCSFPROJ", engine="pyogrio")
print(f"Saved {len(gdf)} features.")


Fetching WFS pages: 47248it [00:18, 2505.48it/s]


Saved 47248 features.


EGIDS abfüllen

In [ ]:
import requests
import geopandas as gpd
import pandas as pd
from io import BytesIO
from tqdm import tqdm
import time

# ── 1. LOAD ORIGINAL GPKG ─────────────────────────────────────────────────────
gdf = gpd.read_file("av_projected_buildings.gpkg", layer="LCSFPROJ", engine="pyogrio")
print(f"Loaded {len(gdf)} features | CRS: {gdf.crs}")
print(f"Columns: {gdf.columns.tolist()}")

# ── 2. EGID COVERAGE CHECK ────────────────────────────────────────────────────
total = len(gdf)
has_egid = gdf["GWR_EGID"].notna() & (gdf["GWR_EGID"] != 0)
print(f"\nEGID coverage: {has_egid.sum()} / {total} ({has_egid.mean()*100:.1f}%)")
print(f"No EGID:        {(~has_egid).sum()} features")
print(f"\nPer canton:\n{gdf.groupby('Kanton')['GWR_EGID'].apply(lambda x: x.notna().sum()).to_string()}")

# ── 3. FETCH GWR ATTRIBUTES FOR ALL EGIDS ────────────────────────────────────
GWR_URL = "https://api3.geo.admin.ch/rest/services/ech/MapServer/ch.bfs.gebaeude_wohnungs_register/{egid}_0"

def fetch_gwr(egid):
    """Fetch full GWR JSON for a single EGID. Returns dict or None."""
    try:
        r = requests.get(GWR_URL.format(egid=int(egid)), timeout=10)
        if r.status_code == 200:
            data = r.json()
            attrs = data.get("feature", {}).get("attributes", {})
            attrs["GWR_EGID"] = int(egid)
            return attrs
    except Exception:
        pass
    return None

# Only fetch rows with a valid EGID
egid_gdf = gdf[has_egid].copy()
egids = egid_gdf["GWR_EGID"].unique()
print(f"\nFetching GWR data for {len(egids)} unique EGIDs...")

results = []
for egid in tqdm(egids):
    row = fetch_gwr(egid)
    if row:
        results.append(row)
    time.sleep(0.05)  # ~20 req/s, be polite

gwr_df = pd.DataFrame(results)
print(f"\nFetched {len(gwr_df)} GWR records")
print(f"GWR columns available:\n{gwr_df.columns.tolist()}")

# ── 4. MERGE GWR ATTRIBUTES INTO GDF ─────────────────────────────────────────
gdf_enriched = gdf.merge(gwr_df, on="GWR_EGID", how="left")

# ── 5. SAVE TO NEW GPKG (ORIGINAL UNTOUCHED) ─────────────────────────────────
gdf_enriched.to_file(
    "av_projected_buildings_enriched.gpkg",
    driver="GPKG",
    layer="LCSFPROJ_GWR",
    engine="pyogrio"
)
print(f"\nSaved enriched GPKG: av_projected_buildings_enriched.gpkg")
print(f"Total columns: {len(gdf_enriched.columns)}")


DataLayerError: Layer 'LCSFPROJ_GWR' could not be opened

In [ ]:
import requests
import geopandas as gpd
import pandas as pd
from io import BytesIO
from tqdm import tqdm
import time

# ── 1. LOAD ORIGINAL GPKG ─────────────────────────────────────────────────────
gdf = gpd.read_file("av_projected_buildings.gpkg", layer="LCSFPROJ", engine="pyogrio")
print(f"Loaded {len(gdf)} features | CRS: {gdf.crs}")

# ── 2. EGID COVERAGE CHECK ────────────────────────────────────────────────────
has_egid = gdf["GWR_EGID"].notna() & (gdf["GWR_EGID"] != 0)
print(f"EGID coverage: {has_egid.sum()} / {len(gdf)} ({has_egid.mean()*100:.1f}%)")

# ── 3. CAST TO SAME TYPE (fixes the merge error) ──────────────────────────────
gdf["GWR_EGID"] = pd.to_numeric(gdf["GWR_EGID"], errors="coerce")

# ── 4. FETCH GWR — TEST WITH 50 EGIDS FIRST ──────────────────────────────────
GWR_URL = "https://api3.geo.admin.ch/rest/services/ech/MapServer/ch.bfs.gebaeude_wohnungs_register/{egid}_0"

def fetch_gwr(egid):
    try:
        r = requests.get(GWR_URL.format(egid=int(egid)), timeout=10)
        if r.status_code == 200:
            data = r.json()
            attrs = data.get("feature", {}).get("attributes", {})
            attrs["GWR_EGID"] = int(egid)
            return attrs
    except Exception as e:
        print(f"Error for EGID {egid}: {e}")
    return None

# ── Grab 50 unique EGIDs for testing ─────────────────────────────────────────
egid_gdf = gdf[has_egid].copy()
TEST_MODE = False          # ← flip to False for full run
test_egids = egid_gdf["GWR_EGID"].dropna().unique()
egids = test_egids[:50] if TEST_MODE else test_egids
print(f"\nRunning in {'TEST (50)' if TEST_MODE else 'FULL'} mode — {len(egids)} EGIDs")

results = []
for egid in tqdm(egids):
    row = fetch_gwr(egid)
    if row:
        results.append(row)
    time.sleep(0.001) # sleep to avoid hitting rate limits (50 req/s max) 

gwr_df = pd.DataFrame(results)

# ── 5. CAST GWR EGID TO INT64 AS WELL ────────────────────────────────────────
gwr_df["GWR_EGID"] = gwr_df["GWR_EGID"].astype("int64")

print(f"\nFetched {len(gwr_df)} GWR records")
print(f"Columns: {gwr_df.columns.tolist()}")

# ── 6. MERGE ──────────────────────────────────────────────────────────────────
gdf_enriched = gdf.merge(gwr_df, on="GWR_EGID", how="left")
print(f"\nEnriched shape: {gdf_enriched.shape}")
print(gdf_enriched[gwr_df.columns].head(5))

# ── 7. SAVE TEST GPKG ─────────────────────────────────────────────────────────
out_file = "av_enriched_TEST.gpkg" if TEST_MODE else "av_projected_buildings_enriched.gpkg"
gdf_enriched.to_file(out_file, driver="GPKG", layer="LCSFPROJ_GWR", engine="pyogrio")
print(f"\nSaved → {out_file}")


DataLayerError: Layer 'LCSFPROJ_GWR' could not be opened

Modify and search for more EGIDS

In [ ]:
import geopandas as gpd
import pandas as pd
import requests
from tqdm import tqdm
import time

# ══════════════════════════════════════════════════════════════════════════════
# CONFIG
# ══════════════════════════════════════════════════════════════════════════════
AV_GPKG     = "av_projected_buildings_enriched.gpkg"
AV_LAYER    = "LCSFPROJ_GWR"
GWR_GEOJSON = "buildings.geojson"
OUT_GPKG    = "av_projected_buildings_enriched_2.gpkg"
GWR_API_URL = "https://api3.geo.admin.ch/rest/services/ech/MapServer/ch.bfs.gebaeude_wohnungs_register/{egid}_0"
TEST_MODE   = False   # ← flip to False for full run

# ══════════════════════════════════════════════════════════════════════════════
# 1. LOAD AV PROJECTED BUILDINGS
# ══════════════════════════════════════════════════════════════════════════════
print("Loading AV buildings...")
gdf = gpd.read_file(AV_GPKG, layer=AV_LAYER, engine="pyogrio")
gdf["GWR_EGID"] = pd.to_numeric(gdf["GWR_EGID"], errors="coerce")
print(f"Total buildings: {len(gdf):,} | CRS: {gdf.crs}")

has_egid    = gdf["GWR_EGID"].notna() & (gdf["GWR_EGID"] != 0)
gdf_without = gdf[~has_egid].copy().reset_index(drop=False).rename(columns={"index": "building_idx"})

if TEST_MODE:
    gdf_without = gdf_without.head(50)
    print(f"TEST MODE: using {len(gdf_without)} buildings")

print(f"With EGID:    {has_egid.sum():,}")
print(f"Without EGID: {len(gdf_without):,}")

# ══════════════════════════════════════════════════════════════════════════════
# 2. LOAD GWR POINTS
# ══════════════════════════════════════════════════════════════════════════════
print("\nLoading GWR points...")
gwr = gpd.read_file(GWR_GEOJSON, engine="pyogrio")
print(f"GWR points loaded: {len(gwr):,}")

if gwr.crs != gdf.crs:
    gwr = gwr.to_crs(gdf.crs)
    print(f"Reprojected GWR to {gdf.crs}")

gwr["_priority"] = 1

# ══════════════════════════════════════════════════════════════════════════════
# 3. SPATIAL JOIN — find GWR points inside buildings WITHOUT EGID
# ══════════════════════════════════════════════════════════════════════════════
print("\nRunning spatial join...")
joined = gpd.sjoin(
    gdf_without[["building_idx", "geometry"]],
    gwr[["geometry", "egid", "_priority"]],
    how="left",
    predicate="intersects"
)

# ══════════════════════════════════════════════════════════════════════════════
# 4. RESOLVE MULTI-EGID CONFLICTS — tiebreak by centroid distance
# ══════════════════════════════════════════════════════════════════════════════
matched = joined.dropna(subset=["egid"]).copy()

building_centroids = gdf_without.set_index("building_idx").geometry.centroid
gwr_geom = gwr.geometry  # keep original index, use .loc

matched["_dist"] = matched.apply(
    lambda row: building_centroids.loc[row["building_idx"]].distance(
        gwr_geom.loc[row["index_right"]]
    ), axis=1
)

best_match = (
    matched
    .sort_values(["building_idx", "_dist"])
    .drop_duplicates(subset="building_idx", keep="first")
)

egid_patch = best_match.set_index("building_idx")["egid"].astype("int64")

print(f"Buildings newly matched spatially: {len(best_match):,}")
print(f"Still unmatched after spatial:     {len(gdf_without) - len(best_match):,}")

# ══════════════════════════════════════════════════════════════════════════════
# 5. FETCH GWR API ATTRIBUTES — NEWLY MATCHED EGIDS ONLY
# ══════════════════════════════════════════════════════════════════════════════
original_egids = set(gdf[has_egid]["GWR_EGID"].dropna().astype("int64").unique())
new_egids      = set(best_match["egid"].dropna().astype("int64").unique())
fetch_egids    = new_egids - original_egids

print(f"Original EGIDs already enriched: {len(original_egids):,}")
print(f"Newly matched EGIDs:             {len(new_egids):,}")
print(f"To fetch:                        {len(fetch_egids):,}")

def fetch_gwr(egid):
    try:
        r = requests.get(GWR_API_URL.format(egid=int(egid)), timeout=10)
        if r.status_code == 200:
            attrs = r.json().get("feature", {}).get("attributes", {})
            attrs["GWR_EGID"] = int(egid)
            return attrs
    except Exception as e:
        print(f"Error {egid}: {e}")
    return None

results = []
for egid in tqdm(fetch_egids):
    row = fetch_gwr(egid)
    if row:
        results.append(row)
    time.sleep(0.05)

gwr_new_df = pd.DataFrame(results)
gwr_new_df["GWR_EGID"] = gwr_new_df["GWR_EGID"].astype("int64")

before = len(gwr_new_df)
gwr_new_df = gwr_new_df[~gwr_new_df["gstat"].isin([1007, 1008])].copy()
print(f"Dropped {before - len(gwr_new_df):,} records with gstat 1007/1008")
print(f"Remaining new GWR records: {len(gwr_new_df):,}")

# ══════════════════════════════════════════════════════════════════════════════
# 6. PATCH NEW EGIDS + ATTRIBUTES BACK INTO ORIGINAL GDF (geometry safe)
# ══════════════════════════════════════════════════════════════════════════════

# Patch new EGID values into the original gdf by position
egid_map = egid_patch.to_dict()  # {building_idx: egid}
for bidx, egid in egid_map.items():
    gdf.loc[bidx, "GWR_EGID"] = egid
gdf["GWR_EGID"] = pd.to_numeric(gdf["GWR_EGID"], errors="coerce")

# Merge new GWR attributes — fills columns for newly matched rows only
gdf_final = gdf.merge(gwr_new_df, on="GWR_EGID", how="left", suffixes=("", "_new"))

# For any duplicate columns, fill original NaN with new values then drop _new
new_cols = [c for c in gdf_final.columns if c.endswith("_new")]
for col in new_cols:
    base = col.replace("_new", "")
    gdf_final[base] = gdf_final[base].combine_first(gdf_final[col])
    gdf_final.drop(columns=col, inplace=True)

print(f"\nGeometry intact: {gdf_final.geometry.notna().all()}")
print(f"Total rows:      {len(gdf_final):,}  (should still be {len(gdf):,})")

# ══════════════════════════════════════════════════════════════════════════════
# 7. SUMMARY
# ══════════════════════════════════════════════════════════════════════════════
total       = len(gdf_final)
final_egids = gdf_final["GWR_EGID"].notna().sum()
gained      = final_egids - has_egid.sum()

print(f"\n{'─'*45}")
print(f"Total buildings:        {total:,}")
print(f"Original EGID coverage: {has_egid.sum():,} ({has_egid.mean()*100:.1f}%)")
print(f"Final EGID coverage:    {final_egids:,} ({final_egids/total*100:.1f}%)")
print(f"Gained spatially:       {gained:,}")
print(f"Still no EGID:          {total - final_egids:,}")
print(f"Columns:                {len(gdf_final.columns)}")
print(f"{'─'*45}")

# ══════════════════════════════════════════════════════════════════════════════
# 8. SAVE
# ══════════════════════════════════════════════════════════════════════════════
out_file = "av_spatial_enriched_3.gpkg" if TEST_MODE else OUT_GPKG
gdf_final.to_file(out_file, driver="GPKG", layer="LCSFPROJ_GWR", engine="pyogrio")
print(f"\nSaved → {out_file}")


Loading AV buildings...
Total buildings: 47,248 | CRS: EPSG:2056
TEST MODE: using 50 buildings
With EGID:    32,400
Without EGID: 50

Loading GWR points...
GWR points loaded: 3,347,244

Running spatial join...
Buildings newly matched spatially: 15
Still unmatched after spatial:     35
Original EGIDs already enriched: 31,970
Newly matched EGIDs:             14
To fetch:                        11


100%|██████████| 11/11 [00:01<00:00,  7.57it/s]


Dropped 0 records with gstat 1007/1008
Remaining new GWR records: 9

Geometry intact: True
Total rows:      47,248  (should still be 47,248)

─────────────────────────────────────────────
Total buildings:        47,248
Original EGID coverage: 32,400 (68.6%)
Final EGID coverage:    32,415 (68.6%)
Gained spatially:       15
Still no EGID:          14,833
Columns:                85
─────────────────────────────────────────────

Saved → av_spatial_enriched_3.gpkg


In [9]:
import geopandas as gpd
gwr_test = gpd.read_file("entrances.geojson", rows=5, engine="pyogrio")
print(gwr_test.columns.tolist())
print(gwr_test.head())

['egid', 'edid', 'streetName', 'entranceNumber', 'zip', 'locality', 'geometry']
   egid  edid streetName entranceNumber   zip            locality  \
0     2     0    Im Feld              1  8910  Affoltern am Albis   
1     3     0    Im Feld              2  8910  Affoltern am Albis   
2     4     0    Im Feld              3  8910  Affoltern am Albis   
3     5     0    Im Feld              4  8910  Affoltern am Albis   
4     6     0    Im Feld              5  8910  Affoltern am Albis   

                          geometry  
0   POINT (2676541.16 1235979.043)  
1  POINT (2676529.561 1236015.749)  
2  POINT (2676510.997 1235979.419)  
3  POINT (2676506.529 1236015.595)  
4    POINT (2676485.838 1235978.3)  


change values

In [4]:
import geopandas as gpd
import pandas as pd

# ── 1. LOAD ENRICHED GPKG ─────────────────────────────────────────────────────
gdf = gpd.read_file("av_projected_buildings_enriched_2.gpkg", layer="LCSFPROJ_GWR", engine="pyogrio")

# ── 2. SELECT & RENAME COLUMNS from key.xlsx ─────────────────────────────────
rename_map = {
    "EPROID":   "Eidgenössischer Bauprojektidentifikator",
    "strname":  "Strassenname",
    "dplz4":    "Postleitzahl",
    "ggdename": "Gemeindename",
    "gstat":    "Gebäudestatus",
    "gkat":     "Gebäudekategorie",
    "garea":    "Gebäudefläche",
    "gvol":     "Gebäudevolumen",
    "gvolsce":  "Informationsquelle zum Gebäudevolumen",
    "gastw":    "Anzahl Geschosse",
    "gebf":     "Energiebezugsfläche",
    "deinr":    "Eingangsnummer Gebäude",
}

# Keep only columns that actually exist in the data
existing_cols = {k: v for k, v in rename_map.items() if k in gdf.columns}
missing_cols  = [k for k in rename_map if k not in gdf.columns]
if missing_cols:
    print(f"Warning – not found in data, skipped: {missing_cols}")

# Always keep geometry + GWR_EGID + Kanton
keep = ["GWR_EGID", "Kanton", "geometry"] + list(existing_cols.keys())
keep = [c for c in keep if c in gdf.columns]
gdf_clean = gdf[keep].copy()
gdf_clean.rename(columns=existing_cols, inplace=True)

# ── 3. MAP gstat CODES → Stand der Arbeit ─────────────────────────────────────
gstat_map = {
    1001: "Projektiert",
    1002: "Bewilligt",
    1003: "Im Bau",
    1004: "Bestehend",
    1005: "Nicht nutzbar",
    1007: "Abgebrochen",
    1008: "Nicht realisiert",
}
gdf_clean["Gebäudestatus"] = (
    pd.to_numeric(gdf_clean["Gebäudestatus"], errors="coerce")
    .map(gstat_map)
)

# ── 4. MAP gkat CODES → Gebäudekategorie ──────────────────────────────────────
gkat_map = {
    1010: "Provisorische Unterkunft",
    1020: "Gebäude mit ausschliesslicher Wohnnutzung",
    1030: "Andere Wohngebäude (Wohngebäude mit Nebennutzung)",
    1040: "Gebäude mit teilweiser Wohnnutzung",
    1060: "Gebäude ohne Wohnnutzung",
    1080: "Sonderbau",
}
gdf_clean["Gebäudekategorie"] = (
    pd.to_numeric(gdf_clean["Gebäudekategorie"], errors="coerce")
    .map(gkat_map)
)

# ── 5. MAP gvolsce CODES → Angaben der Datenquelle ───────────────────────────
gvolsce_map = {
    869: "Gemäss Baubewilligung",
    858: "Gemäss Gebäudeenergieausweis der Kantone (GEAK)",
    853: "Gemäss Gebäudeversicherung",
    852: "Gemäss amtlicher Schätzung",
    857: "Gemäss Eigentümer/in / Verwaltung ",
    851: "Gemäss amtlicher Vermessung ",
    870: "Gemäss topografischem Landschaftsmodell (TLM) ",
    878: "Nicht bestimmbares Volumen (nicht geschlossenes Gebäude) ",
    852: "Gemäss amtlicher Schätzung",
    859: "Andere",
}
gdf_clean["Informationsquelle zum Gebäudevolumen"] = (
    pd.to_numeric(gdf_clean["Informationsquelle zum Gebäudevolumen"], errors="coerce")
    .map(gvolsce_map)
)

# ── 6. QUICK SUMMARY ──────────────────────────────────────────────────────────
print(f"\nFinal columns:\n{gdf_clean.columns.tolist()}")
print(f"\nGebäudestatus distribution:\n{gdf_clean['Gebäudestatus'].value_counts(dropna=False)}")
print(f"\nGebäudekategorie distribution:\n{gdf_clean['Gebäudekategorie'].value_counts(dropna=False)}")
print(f"\nDatenquelle Volumen:\n{gdf_clean['Informationsquelle zum Gebäudevolumen'].value_counts(dropna=False)}")

# # ── 6b. INTERACTIVELY DROP UNWANTED COLUMNS ───────────────────────────────────
# print("\nCurrent columns (excluding geometry):")
# non_geom = [c for c in gdf_clean.columns if c != "geometry"]
# for i, col in enumerate(non_geom):
#     print(f"  [{i}] {col}")

# drop_input = input("\nEnter column numbers to DROP (comma-separated), or press Enter to keep all: ")

# if drop_input.strip():
#     drop_indices = [int(x.strip()) for x in drop_input.split(",")]
#     drop_cols = [non_geom[i] for i in drop_indices]
#     gdf_clean.drop(columns=drop_cols, inplace=True)
#     print(f"\nDropped: {drop_cols}")
# else:
#     print("No columns dropped.")

# print(f"\nRemaining columns: {gdf_clean.columns.tolist()}")

# ── 7. SAVE TO NEW GPKG ───────────────────────────────────────────────────────
gdf_clean.to_file(
    "av_projected_buildings_final.gpkg",
    driver="GPKG",
    layer="LCSFPROJ_final",
    engine="pyogrio"
)
print("\nSaved → av_projected_buildings_final.gpkg")


Warning – not found in data, skipped: ['EPROID']

Final columns:
['GWR_EGID', 'Kanton', 'geometry', 'Strassenname', 'Postleitzahl', 'Gemeindename', 'Gebäudestatus', 'Gebäudekategorie', 'Gebäudefläche', 'Gebäudevolumen', 'Informationsquelle zum Gebäudevolumen', 'Anzahl Geschosse', 'Energiebezugsfläche', 'Eingangsnummer Gebäude']

Gebäudestatus distribution:
Gebäudestatus
NaN            14232
Bestehend      11573
Bewilligt       9572
Im Bau          7429
Projektiert     4442
Name: count, dtype: int64

Gebäudekategorie distribution:
Gebäudekategorie
Gebäude mit ausschliesslicher Wohnnutzung            21830
NaN                                                  14232
Gebäude ohne Wohnnutzung                              7601
Andere Wohngebäude (Wohngebäude mit Nebennutzung)     1791
Sonderbau                                             1141
Gebäude mit teilweiser Wohnnutzung                     641
Provisorische Unterkunft                                12
Name: count, dtype: int64

Datenqu